# Phase 2 long-T step-10 reproducer (Colab A100)

Runs `experiments/phase2_step10_long_t.py` to sweep
T ∈ {10k, 100k, 1M, 10M} at N=5 seeds × 4 conditions, confirming
step 10's headline number holds at long training durations before
Block 9 commits N=500 cycles.

**Before running:** Runtime → Change runtime type → **A100 GPU**
(Colab Pro+). T=10M will fail on smaller GPUs.

**Expected wall-time:** ~90 min total at T=10M on A100. The
10k/100k/1M sweeps complete in the first few minutes; T=10M
dominates.

**Pro+ background execution:** the runtime keeps running for up
to 24 hr even if you close the browser tab. Reopen the notebook
later to see results. Cell 7 saves to Drive so the output is
durable across runtime resets.

## 1. Confirm A100 attached

Should show `NVIDIA A100-SXM4-40GB` (or 80GB on some Pro+ tiers).
If it shows T4/V100/L4, change runtime type and re-execute.

In [ ]:
!nvidia-smi

## 2. Clone silicritter and install

Public repo, no auth needed. `pip install -e .` registers the
package in editable mode and pulls JAX with CUDA12 support.

In [ ]:
!git clone --depth 1 https://github.com/edhodapp/silicritter.git
%cd silicritter
!pip install -q -e ".[dev]"

## 3. Verify JAX sees the A100

Should print `JAX backend: gpu` and a device list containing
`CudaDevice(id=0)` (or similar). If `cpu`, JAX didn't find the
GPU; check that the runtime type is set to A100.

In [ ]:
import jax
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")
assert jax.default_backend() == 'gpu', 'Need GPU runtime; switch via Runtime > Change runtime type'

## 4. Sanity-check the test suite

Runs the full pytest suite (~3 min on A100) before committing
to the 90-min Phase 2 run. If anything is broken in the Colab
environment (CUDA mismatch, JAX version drift, etc.), this
fails fast instead of mid-run.

In [ ]:
!python -m pytest tests/ -q --tb=short -x

## 5. Run Phase 2 (the big one)

Tee'd to `phase2_run.log` so you can see live progress in
the cell output AND have a copy on disk if the cell scrolls
off-screen. `python -u` for unbuffered stdout so you see
lines as they're emitted, not in batched chunks.

In [ ]:
!python -u experiments/phase2_step10_long_t.py 2>&1 | tee phase2_run.log

## 6. Display results inline

Outputs the CSV + log inline. Total ~12 KB. Useful for a quick
look while the kernel is still alive; cell 7 makes a durable
Drive copy.

In [ ]:
print('=' * 60)
print('CSV: overnight_results/phase2_step10_long_t.csv')
print('=' * 60)
!cat overnight_results/phase2_step10_long_t.csv
print()
print('=' * 60)
print('LOG: overnight_results/phase2_step10_long_t.log')
print('=' * 60)
!cat overnight_results/phase2_step10_long_t.log

## 7. Save results to Drive (durable copy)

Mounts your Google Drive and copies the CSV + both logs to
`MyDrive/silicritter_phase2/<timestamp>/`. First run prompts
for Google account auth in a popup; subsequent runs reuse the
mount. After this cell finishes, the files are durable across
runtime resets and browseable at drive.google.com.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime, timezone
stamp = datetime.now(tz=timezone.utc).strftime('%Y-%m-%dT%H%M%SZ')
dest = f'/content/drive/MyDrive/silicritter_phase2/{stamp}'
!mkdir -p "$dest"
!cp overnight_results/phase2_step10_long_t.csv "$dest/"
!cp overnight_results/phase2_step10_long_t.log "$dest/"
!cp phase2_run.log "$dest/"
print(f'Saved to: {dest}')